# Sparkle V2 — Fase 3.0: Binarización del target + re-split por participante

**Proyecto:** Sparkle V2 — Módulo de detección de desatención (Boredom, DAiSEE)
**Entorno de ejecución:** Google Colab (GPU T4) o Kaggle Notebooks (GPU)
**Punto de partida:** BiGRU v2 (Fase 2) — Test accuracy (3 niveles) ≈ trivial 45,6%.
**Objetivo de este notebook:** ejecutar el **Roadmap Fase 3.0** de `Plan_Fase3_Deteccion_Desatencion_70pct.md`:

1. **Paso A** — Binarizar Boredom ({0}=Atento vs {1,2,3}=Desatención) y re-entrenar el BiGRU actual **tal cual**, sobre el split oficial. Valida empíricamente la hipótesis de "desplazamiento superficial" (§1.1 del plan). Resultado esperado: ~52–56% de accuracy, sin ganancia real más allá del corrimiento del trivial (54,4%).
2. **Paso B** — Re-split agrupado por participante (`StratifiedGroupKFold`, Test oficial intacto) + ventanas deslizantes (augmentación) + label smoothing, y reentrenamiento del BiGRU binario.

**Criterio de salida de Fase 3.0:** brecha vs. trivial > +3 pts de forma robusta (media ± desvío de 5 semillas). Resultado esperado del plan: 57–61% de accuracy.

> Este notebook asume que `features_v2/{Train,Validation,Test}/*.npz` y `dataset_final/dataset_v2.npz` (generados en Fase 2) ya existen en `PROJECT_DIR` (Google Drive o Kaggle dataset). No vuelve a tocar video ni requiere GPU real para el volumen de datos actual, pero se acelera notablemente con GPU.

In [ ]:
# Colab: Runtime -> Change runtime type -> GPU (T4)
!pip install -q scikit-learn tensorflow

import os, glob, json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from collections import Counter
from sklearn.metrics import (accuracy_score, f1_score, balanced_accuracy_score,
                              cohen_kappa_score, classification_report, confusion_matrix)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler

print("TF:", tf.__version__, "| GPU:", tf.config.list_physical_devices('GPU'))

# --- Descomentar en Colab ---
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT_DIR = '/content/drive/MyDrive/SparkleV2'

# --- Descomentar en Kaggle (dataset propio subido como 'sparklev2') ---
# PROJECT_DIR = '/kaggle/input/sparklev2'

PROJECT_DIR  = '/content/drive/MyDrive/SparkleV2'   # <-- ajustar segun entorno
FEATURES_DIR = f'{PROJECT_DIR}/features_v2'
DATASET_V2   = f'{PROJECT_DIR}/dataset_final/dataset_v2.npz'
OUT_DIR      = f'{PROJECT_DIR}/dataset_final_fase3'
CKPT_DIR     = f'{PROJECT_DIR}/checkpoints_fase3'
os.makedirs(OUT_DIR,  exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)
N_FRAMES = 60

## Paso A — Binarización sobre el split oficial (sin re-split)

Objetivo: aislar el efecto puro de la binarización, sin mezclar con la mitigación de Eje 4. Se reutiliza `dataset_v2.npz` (ya consolidado en Fase 2: NaN tratado, deltas, escalado con Train oficial) y solo se remapea el target.

In [ ]:
d = np.load(DATASET_V2, allow_pickle=True)
X_train, y_train_raw = d['X_train'], d['y_train']
X_val,   y_val_raw   = d['X_val'],   d['y_val']
X_test,  y_test_raw  = d['X_test'],  d['y_test']
FEATS_FULL = list(d['feature_names'])

def binarize(y):
    # Boredom {0} -> Atento (0) ; {1,2,3} -> Desatencion (1)
    return (y > 0).astype(np.int64)

y_train, y_val, y_test = binarize(y_train_raw), binarize(y_val_raw), binarize(y_test_raw)

for name, y in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    c = Counter(y.tolist())
    print(f"{name}: n={len(y)} | Atento={c.get(0,0)} ({c.get(0,0)/len(y):.3f}) "
          f"| Desatencion={c.get(1,0)} ({c.get(1,0)/len(y):.3f})")

trivial_test = max(Counter(y_test.tolist()).values()) / len(y_test)
print(f"\nTrivial (Test, split oficial) = {trivial_test:.4f}")
# Esperado ~0.544 (891/1638) segun el diagnostico del plan (S1.1)

In [ ]:
def smoothed_class_weights(y, beta=0.5):
    counts = Counter(y.tolist()); total = len(y)
    w = {int(c): (total / counts[c]) ** beta for c in counts}
    mean_w = np.mean(list(w.values()))
    return {c: w[c] / mean_w for c in w}

def build_gru_binary(input_shape, units=32, dropout=0.4, l2=1e-4, lr=5e-4,
                      label_smoothing=0.0):
    reg = keras.regularizers.l2(l2)
    inp = keras.Input(shape=input_shape)
    x = layers.Masking(mask_value=0.0)(inp)
    x = layers.Bidirectional(
            layers.GRU(units, dropout=dropout, recurrent_dropout=0.2,
                       kernel_regularizer=reg))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(16, activation='relu', kernel_regularizer=reg)(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    model = keras.Model(inp, out)
    model.compile(
        optimizer=keras.optimizers.Adam(lr, clipnorm=1.0),
        loss=keras.losses.BinaryCrossentropy(label_smoothing=label_smoothing),
        metrics=['accuracy'])
    return model

class MacroF1Binary(keras.callbacks.Callback):
    def __init__(self, val_data):
        super().__init__(); self.Xv, self.yv = val_data
    def on_epoch_end(self, epoch, logs=None):
        p = (self.model.predict(self.Xv, verbose=0).ravel() > 0.5).astype(int)
        logs['val_macro_f1'] = f1_score(self.yv, p, average='macro')

def train_and_eval(X_tr, y_tr, X_va, y_va, X_te, y_te, seed, class_weight,
                    label_smoothing=0.0, epochs=120, patience=15, tag='pasoA'):
    tf.random.set_seed(seed); np.random.seed(seed)
    model = build_gru_binary(X_tr.shape[1:], label_smoothing=label_smoothing)
    model.fit(
        X_tr, y_tr, validation_data=(X_va, y_va),
        epochs=epochs, batch_size=64, class_weight=class_weight,
        callbacks=[
            MacroF1Binary((X_va, y_va)),
            keras.callbacks.EarlyStopping(monitor='val_macro_f1', mode='max',
                                          patience=patience, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(monitor='val_macro_f1', mode='max',
                                              factor=0.5, patience=6, min_lr=1e-5),
        ], verbose=0)
    p_te = (model.predict(X_te, verbose=0).ravel() > 0.5).astype(int)
    return model, {
        'seed': seed,
        'acc': accuracy_score(y_te, p_te),
        'f1_macro': f1_score(y_te, p_te, average='macro'),
        'balanced_acc': balanced_accuracy_score(y_te, p_te),
        'kappa': cohen_kappa_score(y_te, p_te),
    }, p_te

In [ ]:
class_weight_A = smoothed_class_weights(y_train, beta=0.5)
print('class_weight (Paso A):', class_weight_A)

SEEDS = [0, 1, 2, 3, 4]
results_A = []
best_model_A, best_p_A = None, None
for seed in SEEDS:
    model, r, p_te = train_and_eval(X_train, y_train, X_val, y_val, X_test, y_test,
                                     seed, class_weight_A, label_smoothing=0.0,
                                     epochs=120, patience=15, tag='pasoA')
    r['gap_vs_trivial'] = r['acc'] - trivial_test
    results_A.append(r)
    print(f"seed={seed}: acc={r['acc']:.4f} f1_macro={r['f1_macro']:.4f} "
          f"bal_acc={r['balanced_acc']:.4f} gap={r['gap_vs_trivial']:+.4f}")
    if seed == SEEDS[0]:
        best_model_A, best_p_A = model, p_te

df_A = pd.DataFrame(results_A)
df_A.to_csv(f'{CKPT_DIR}/paso_a_resultados_5semillas.csv', index=False)

print('\n=== RESUMEN PASO A (5 semillas) ===')
print(f"Accuracy Test:   {df_A['acc'].mean():.4f} +/- {df_A['acc'].std():.4f}  (trivial={trivial_test:.4f})")
print(f"F1 macro Test:   {df_A['f1_macro'].mean():.4f} +/- {df_A['f1_macro'].std():.4f}")
print(f"Balanced acc:    {df_A['balanced_acc'].mean():.4f} +/- {df_A['balanced_acc'].std():.4f}")
print(f"Gap vs trivial:  {df_A['gap_vs_trivial'].mean():+.4f} +/- {df_A['gap_vs_trivial'].std():.4f}")
print(classification_report(y_test, best_p_A, target_names=['Atento','Desatencion'], digits=3))

**Lectura del Paso A:** si la brecha media vs. trivial queda por debajo de +3 pts (rango ~52–56% esperado por el plan), confirma que la binarización sola **no alcanza** el 70% — es el "desplazamiento superficial" documentado en §1.1. Esto no es un resultado negativo: es la evidencia que sustenta pasar a la palanca de señal real (Eje 2, Fase 3.2) y, mientras tanto, a la mitigación de protocolo del Paso B.

## Paso B — Re-split agrupado por participante + augmentación + label smoothing

Se reconstruyen las secuencias **crudas** de `features_v2/{Train,Validation}` (mismo tratamiento de NaN y deltas del notebook de Fase 2), se fusionan y se re-particionan con `StratifiedGroupKFold` agrupando por participante (primeros 6 dígitos del `ClipID`). El **Test oficial permanece intacto** — nunca se re-mezcla, para mantener comparabilidad con la literatura y con el Paso A.

In [ ]:
DELTA_FEATURES = ['ear_avg', 'yaw', 'pitch', 'mar', 'jawOpen',
                  'eyeBlinkLeft', 'eyeBlinkRight',
                  'eyeLookInLeft', 'eyeLookOutLeft', 'browInnerUp']

def load_raw_split(split):
    files = sorted(glob.glob(f'{FEATURES_DIR}/{split}/*.npz'))
    X, y3, ids = [], [], []
    feat_names = None
    for f in files:
        dd = np.load(f, allow_pickle=True)
        seq = dd['sequence'].astype(np.float32)
        if seq.shape != (N_FRAMES, 22):
            continue
        if feat_names is None:
            feat_names = list(dd['feature_names'])
        X.append(seq)
        b = int(dd['boredom'])
        y3.append(0 if b == 0 else (1 if b == 1 else 2))   # mismo collapse_boredom de Fase 2
        ids.append(os.path.splitext(os.path.basename(f))[0])
    return np.stack(X), np.array(y3, dtype=np.int64), np.array(ids), feat_names

X_tr_raw, y3_tr, id_tr, FEATS22 = load_raw_split('Train')
X_va_raw, y3_va, id_va, _       = load_raw_split('Validation')

# Fusion Train+Validation (Test oficial se deja aparte, se carga por separado abajo)
X_pool_raw = np.concatenate([X_tr_raw, X_va_raw], axis=0)
y3_pool    = np.concatenate([y3_tr, y3_va])
id_pool    = np.concatenate([id_tr, id_va])
participant_pool = np.array([cid[:6] for cid in id_pool])

y_bin_pool = (y3_pool > 0).astype(np.int64)
n_participants = len(set(participant_pool.tolist()))
print(f'Pool Train+Validation: {len(id_pool)} clips, {n_participants} participantes')
print('Distribucion binaria pool:', Counter(y_bin_pool.tolist()))

In [ ]:
def interpolate_sequence(seq):
    df = pd.DataFrame(seq)
    df = df.interpolate(method='linear', axis=0, limit_direction='both')
    df = df.ffill().bfill()
    return df.values.astype(np.float32)

def clean_raw(X, train_medians=None, nan_frame_thresh=0.30):
    keep, X_clean = [], []
    for i in range(len(X)):
        frac_nan = np.isnan(X[i]).any(axis=1).mean()
        if frac_nan > nan_frame_thresh:
            continue
        seq = interpolate_sequence(X[i])
        if np.isnan(seq).any() and train_medians is not None:
            idx = np.where(np.isnan(seq))
            seq[idx] = np.take(train_medians, idx[1])
        X_clean.append(seq); keep.append(i)
    return np.stack(X_clean), np.array(keep)

delta_idx = [FEATS22.index(f) for f in DELTA_FEATURES]
def add_deltas(X, idx):
    d_ = np.zeros_like(X[:, :, idx])
    d_[:, 1:, :] = np.diff(X[:, :, idx], axis=1)
    return np.concatenate([X, d_], axis=-1)

FEATS_FULL_B = FEATS22 + [f'd_{f}' for f in DELTA_FEATURES]

def sliding_windows(X, y, win_lens=(40, 44, 48), stride=8, n_frames=60):
    """Ventanas deslizantes SOLO para Train de cada fold. Cada sub-ventana se
    rellena con ceros hasta 60 pasos (consistente con el Masking del modelo)."""
    Xw, yw = [], []
    for i in range(len(X)):
        for L in win_lens:
            for start in range(0, n_frames - L + 1, stride):
                sub = X[i, start:start+L, :]
                pad = np.zeros((n_frames - L, X.shape[-1]), dtype=X.dtype)
                Xw.append(np.concatenate([sub, pad], axis=0))
                yw.append(y[i])
    return np.stack(Xw), np.array(yw)

In [ ]:
LABEL_SMOOTHING = 0.1
N_FOLDS = 5
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

cv_results = []
for fold, (tr_idx, va_idx) in enumerate(sgkf.split(X_pool_raw, y_bin_pool, groups=participant_pool)):
    # 1) NaN + deltas (medianas calculadas SOLO con el train de este fold)
    train_medians = np.nanmedian(X_pool_raw[tr_idx].reshape(-1, 22), axis=0)
    X_tr_c, keep_tr = clean_raw(X_pool_raw[tr_idx], train_medians)
    X_va_c, keep_va = clean_raw(X_pool_raw[va_idx], train_medians)
    y_tr_f = y_bin_pool[tr_idx][keep_tr]
    y_va_f = y_bin_pool[va_idx][keep_va]

    X_tr_d = add_deltas(X_tr_c, delta_idx)
    X_va_d = add_deltas(X_va_c, delta_idx)

    # 2) Scaler fit SOLO en train del fold
    scaler = StandardScaler().fit(X_tr_d.reshape(-1, X_tr_d.shape[-1]))
    def scale(X):
        N, T, F = X.shape
        return scaler.transform(X.reshape(-1, F)).reshape(N, T, F).astype(np.float32)
    X_tr_s, X_va_s = scale(X_tr_d), scale(X_va_d)

    # 3) Ventanas deslizantes SOLO en train
    X_tr_aug, y_tr_aug = sliding_windows(X_tr_s, y_tr_f)
    print(f'Fold {fold}: train {len(X_tr_s)} -> {len(X_tr_aug)} (aug) | val {len(X_va_s)} '
          f'| participantes train={len(set(participant_pool[tr_idx]))} val={len(set(participant_pool[va_idx]))}')

    cw = smoothed_class_weights(y_tr_aug, beta=0.5)
    _, r, _ = train_and_eval(X_tr_aug, y_tr_aug, X_va_s, y_va_f, X_va_s, y_va_f,
                              seed=fold, class_weight=cw, label_smoothing=LABEL_SMOOTHING,
                              epochs=120, patience=15, tag=f'pasoB_fold{fold}')
    # (Nota: aqui X_te=X_va del fold porque el Test oficial NO se toca en la seleccion de folds;
    #  la evaluacion sobre Test oficial se hace una sola vez, en la celda final, con el modelo
    #  entrenado sobre el re-split completo.)
    r['fold'] = fold
    cv_results.append(r)
    print(f"  fold {fold}: val_acc={r['acc']:.4f} val_f1_macro={r['f1_macro']:.4f} "
          f"val_bal_acc={r['balanced_acc']:.4f}")

df_cv = pd.DataFrame(cv_results)
df_cv.to_csv(f'{CKPT_DIR}/paso_b_cv_5folds.csv', index=False)
print('\n=== RESUMEN CV POR PARTICIPANTE (5 folds) ===')
print(f"acc:      {df_cv['acc'].mean():.4f} +/- {df_cv['acc'].std():.4f}")
print(f"f1_macro: {df_cv['f1_macro'].mean():.4f} +/- {df_cv['f1_macro'].std():.4f}")
print(f"bal_acc:  {df_cv['balanced_acc'].mean():.4f} +/- {df_cv['balanced_acc'].std():.4f}")

### Modelo final del Paso B — entrenado sobre todo el re-split, evaluado UNA vez sobre Test oficial

Con la robustez de la CV ya documentada arriba, se entrena un modelo final sobre **todo** el pool re-particionado (Train+Validation fusionados) y se evalúa **una única vez** sobre el Test oficial (nunca visto en ningún fold), que es el número comparable con el Paso A y con Fase 2.

In [ ]:
# Limpieza + deltas + scaler sobre TODO el pool (Train+Validation fusionados)
train_medians_full = np.nanmedian(X_pool_raw.reshape(-1, 22), axis=0)
X_pool_c, keep_pool = clean_raw(X_pool_raw, train_medians_full)
y_pool_f = y_bin_pool[keep_pool]
X_pool_d = add_deltas(X_pool_c, delta_idx)

scaler_final = StandardScaler().fit(X_pool_d.reshape(-1, X_pool_d.shape[-1]))
def scale_final(X):
    N, T, F = X.shape
    return scaler_final.transform(X.reshape(-1, F)).reshape(N, T, F).astype(np.float32)
X_pool_s = scale_final(X_pool_d)
X_pool_aug, y_pool_aug = sliding_windows(X_pool_s, y_pool_f)

# Test oficial: mismo tratamiento (NaN con medianas del pool, deltas, escalado con scaler_final)
X_te_raw, y3_te, id_te, _ = load_raw_split('Test')
y_bin_te = (y3_te > 0).astype(np.int64)
X_te_c, keep_te = clean_raw(X_te_raw, train_medians_full)
y_te_f = y_bin_te[keep_te]
X_te_d = add_deltas(X_te_c, delta_idx)
X_te_s = scale_final(X_te_d)

trivial_test_official = max(Counter(y_te_f.tolist()).values()) / len(y_te_f)
print(f'Test oficial (re-split): n={len(y_te_f)} | trivial={trivial_test_official:.4f}')

cw_final = smoothed_class_weights(y_pool_aug, beta=0.5)
final_results = []
best_final_model, best_final_p = None, None
for seed in SEEDS:
    model, r, p_te = train_and_eval(X_pool_aug, y_pool_aug, X_te_s, y_te_f, X_te_s, y_te_f,
                                     seed, cw_final, label_smoothing=LABEL_SMOOTHING,
                                     epochs=120, patience=15, tag='pasoB_final')
    r['gap_vs_trivial'] = r['acc'] - trivial_test_official
    final_results.append(r)
    print(f"seed={seed}: acc={r['acc']:.4f} f1_macro={r['f1_macro']:.4f} gap={r['gap_vs_trivial']:+.4f}")
    if seed == SEEDS[0]:
        best_final_model, best_final_p = model, p_te

df_final = pd.DataFrame(final_results)
df_final.to_csv(f'{CKPT_DIR}/paso_b_final_5semillas.csv', index=False)
best_final_model.save(f'{CKPT_DIR}/bigru_binario_pasoB_seed0.keras')

print('\n=== RESUMEN PASO B — MODELO FINAL SOBRE TEST OFICIAL (5 semillas) ===')
print(f"Accuracy Test:  {df_final['acc'].mean():.4f} +/- {df_final['acc'].std():.4f}  "
      f"(trivial={trivial_test_official:.4f})")
print(f"F1 macro Test:  {df_final['f1_macro'].mean():.4f} +/- {df_final['f1_macro'].std():.4f}")
print(f"Gap vs trivial: {df_final['gap_vs_trivial'].mean():+.4f} +/- {df_final['gap_vs_trivial'].std():.4f}")
print(classification_report(y_te_f, best_final_p, target_names=['Atento','Desatencion'], digits=3))

cm = confusion_matrix(y_te_f, best_final_p, normalize='true')
import matplotlib.pyplot as plt
plt.imshow(cm); plt.colorbar()
plt.xticks([0,1], ['Atento','Desatencion'], rotation=45); plt.yticks([0,1], ['Atento','Desatencion'])
plt.title('Matriz de confusion (normalizada) - Test oficial - Paso B'); plt.tight_layout()
plt.savefig(f'{CKPT_DIR}/paso_b_confusion_matrix.png', dpi=120)
plt.show()

## Criterio de salida de Fase 3.0

| Paso | Métrica | Umbral del plan | Resultado |
| :--- | :--- | :--- | :--- |
| A (binarización, split oficial) | Accuracy Test | ~52–56% esperado | ver `paso_a_resultados_5semillas.csv` |
| B (re-split + aug + label smoothing) | Brecha vs. trivial (media 5 semillas) | > +3 pts de forma robusta | ver `paso_b_final_5semillas.csv` |
| B | Accuracy Test | 57–61% esperado | ver arriba |

Si el Paso B cumple el criterio de salida, corresponde avanzar a **Fase 3.1** (TCN + multi-task + cabeza ordinal CORAL, ver `Plan_Fase3_Deteccion_Desatencion_70pct.md` §3 y Roadmap). Si no lo cumple con claridad, revisar antes de continuar: (a) balance del sampler por clase, (b) `beta` del class weighting (barrer {0, 0.25, 0.5, 0.75, 1.0} como en Fase 2), (c) si la augmentación por ventanas está introduciendo ruido de padding excesivo (probar `win_lens` más largos, p. ej. (48, 52, 56)).